# Análise Exploratória de Dados (EDA) - Tiny Dataset
Vamos analisar os dados fornecidos (data_tiny) para o distrito de Braga.
O objetivo é carregar os CSVs, validar a integridade das janelas temporais, medir pesos face às capacidades e mapear geograficamente as paragens.

In [7]:
%pip install pandas folium

Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd
import folium

# Carregar os datasets a partir da pasta 'data_tiny'
pasta = "data_tiny/"
df_localities = pd.read_csv(pasta + "localities.csv")
df_locations = pd.read_csv(pasta + "locations.csv")
df_producers = pd.read_csv(pasta + "producers.csv")
df_products = pd.read_csv(pasta + "products.csv")
df_orders = pd.read_csv(pasta + "orders.csv")
df_order_items = pd.read_csv(pasta + "order_items.csv")
df_vehicles = pd.read_csv(pasta + "vehicles.csv")

print("Todos os ficheiros CSV foram carregados com sucesso da pasta data_tiny!")

Todos os ficheiros CSV foram carregados com sucesso da pasta data_tiny!


## 1. Validação de Janelas Temporais
Vamos garantir que a disponibilidade do produto não termina antes de começar, e que a entrega é feita após o produto estar disponível para recolha.

In [9]:
#Verificar se as janelas de disponibilidade são válidas e que as entregas ocorrem após a disponibilidade
colunas_data = ['availability_start', 'availability_end', 'delivery_start', 'delivery_end']
for col in colunas_data:
    df_order_items[col] = pd.to_datetime(df_order_items[col])

valid_avail = (df_order_items['availability_end'] >= df_order_items['availability_start']).all()
valid_deliv = (df_order_items['delivery_end'] >= df_order_items['availability_start']).all()

print(f"As janelas de disponibilidade são logicamente válidas? {valid_avail}")
print(f"As entregas ocorrem após a disponibilidade do produto? {valid_deliv}")

As janelas de disponibilidade são logicamente válidas? True
As entregas ocorrem após a disponibilidade do produto? True


## 2. Pesos vs Capacidades dos Veículos
Temos de garantir que as encomendas cabem nos veículos.

In [ ]:
df_pesos = pd.merge(df_order_items, df_products, on='product_id')
df_pesos['peso_total'] = df_pesos['quantity'] * df_pesos['unit_weight_kg']

peso_maximo_item = df_pesos['peso_total'].max()
capacidade_minima_veiculo = df_vehicles['capacity_kg'].min()

print(f"Item mais pesado: {peso_maximo_item:.2f} kg")
print(f"Veículo com menor capacidade suporta: {capacidade_minima_veiculo:.2f} kg")

#Garantir que as encomendas cabem no veículo em que a encomenda será transportada
if peso_maximo_item <= capacidade_minima_veiculo:
    print("Todas as encomendas individuais cabem em qualquer veículo da frota.")
else:
    print("AVISO: Existem itens que ultrapassam a capacidade!")

Item mais pesado: 24.60 kg
Veículo com menor capacidade suporta: 1009.80 kg
OK: Todas as encomendas individuais cabem em qualquer veículo da frota.


## 3. Mapeamento das Paragens
Visualização dos produtores, clientes e centros de recolha no distrito de Braga.

In [11]:
mapa = folium.Map(location=[41.55, -8.4], zoom_start=10)

#Cores que os ponto irão receber no mapa
cores = {
    'producer': 'green', 
    'collection_center': 'red', 
    'delivery_location': 'blue', 
    'customer': 'blue', 
    'delivery': 'blue', 
    'center': 'red'
}

for index, row in df_locations.iterrows():
    tipo = row['location_type']
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"{row['name']} ({tipo})",
        icon=folium.Icon(color=cores.get(tipo, 'gray'), icon='info-sign')
    ).add_to(mapa)

In [12]:
# Mostra o mapa
mapa